In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(2)
np.set_printoptions(precision=6, suppress=True)

## Ejercicio 2

In [2]:
nn.LSTM?

Init signature: nn.LSTM(*args, **kwargs)
Docstring:     
__init__(input_size,hidden_size,num_layers=1,bias=True,batch_first=False,dropout=0.0,bidirectional=False,proj_size=0,device=None,dtype=None)

Apply a multi-layer long short-term memory (LSTM) RNN to an input sequence.
For each element in the input sequence, each layer computes the following
function:

.. math::
    \begin{array}{ll} \\
        i_t = \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) \\
        f_t = \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) \\
        g_t = \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) \\
        o_t = \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) \\
        c_t = f_t \odot c_{t-1} + i_t \odot g_t \\
        h_t = o_t \odot \tanh(c_t) \\
    \end{array}

where :math:`h_t` is the hidden state at time `t`, :math:`c_t` is the cell
state at time `t`, :math:`x_t` is the input at time `t`, :math:`h_{t-1}`
is the hidden state of the layer at time `t-1` or the initial hidden

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(26, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, u):
        h = torch.relu(self.fc1(u))
        return self.fc2(h)

def evaluar(u, labels):
    modelo.eval()
    with torch.no_grad():
        logits = modelo(u)
        loss = criterio(logits, labels).item()
        acc = (logits.argmax(dim=1) == labels).float().mean().item()
    return loss, acc

In [ ]:
u = torch.tensor([[1., 2.] + [0.] * 24], dtype=torch.float32)
labels = torch.tensor([1], dtype=torch.long)
modelo = MLP()
with torch.no_grad():
    for parametro in modelo.parameters():
        parametro.zero_()
    modelo.fc1.weight[:2, :2] = ### <-- COMPLETAR
    modelo.fc1.bias[2:] = ### <-- COMPLETAR
    modelo.fc2.weight[:2, :2] = ### <-- COMPLETAR
    modelo.fc2.bias[2:] = ### <-- COMPLETAR

criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.SGD(modelo.parameters(), lr=...) ### <-- COMPLETAR
fila1_inicial = modelo.fc1.weight[0].detach().clone()
print('Dimensiones de u:', tuple(u.shape))

In [ ]:
modelo.train()
logits = modelo(u)
p_inicial = ### <-- COMPLETAR
loss = criterio(logits, labels)
loss_inicial = loss.item()
print('Probabilidades iniciales:', p_inicial)
print('Probabilidad de la clase correcta:', p_inicial[1])
print('Predicción (clases 1-10):', np.argmax(p_inicial) + 1)
print('Entropía cruzada inicial:', loss_inicial)

In [ ]:
optimizador.zero_grad()
loss ### <-- COMPLETAR
grad_fc1 = modelo.fc1.weight.grad[:2].detach().clone().numpy()
normas_gradientes = ### <-- COMPLETAR
with torch.no_grad():
    preactivaciones = ### <-- COMPLETAR
print('Preactivaciones de las primeras dos neuronas:', preactivaciones)
print('Primeras dos columnas de sus gradientes:')
print(grad_fc1[:, :2])
print('Normas de las filas:', normas_gradientes)

In [ ]:
optimizador.step()
loss_despues, _ = ### <-- COMPLETAR
print('Pérdida antes y después:', loss_inicial, loss_despues)
print('Cambio máximo en la primera fila de W1:', torch.max(torch.abs(modelo.fc1.weight[0] - fila1_inicial)).item())

## Ejercicio 4

In [ ]:
class CNNBiLSTM(nn.Module):
    def __init__(self, n_clases):
        super().__init__()
        self.conv = nn.Conv2d(1, 8, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))
        self.lstm = nn.LSTM(input_size=160, hidden_size=32, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(64, n_clases)

    def forward(self, u):
        z = torch.relu(self.conv(u))
        z = self.pool(z)
        batch_size, n_canales, n_filtros_mel, n_frames = z.shape
        z = z.permute(0, 3, 1, 2)
        z = z.reshape(batch_size, n_frames, n_canales * n_filtros_mel)
        h, _ = self.lstm(z)
        resumen = h.mean(dim=1)
        return self.fc(resumen)

In [ ]:
torch.manual_seed(42)
modelo_cnn = CNNBiLSTM(n_clases=4)
modelo_cnn.eval()
u_a = torch.zeros((1, 1, 40, 4))
u_b = u_a.clone()
u_b[:, :, :, -1] = ### <-- COMPLETAR

def salidas_intermedias(modelo_cnn, u):
    z = torch.relu(modelo_cnn.conv(u))
    z = modelo_cnn.pool(z)
    batch_size, n_canales, n_filtros_mel, n_frames = z.shape
    z = z.permute(0, 3, 1, 2)
    z = z.reshape(batch_size, n_frames, n_canales * n_filtros_mel)
    h, _ = modelo_cnn.lstm(z)
    return z, h

In [ ]:
with torch.no_grad():
    z_a, h_a = ### <-- COMPLETAR
    z_b, h_b = ### <-- COMPLETAR
    dif_adelante = ### <-- COMPLETAR
    dif_atras = ### <-- COMPLETAR
print('Dimensiones de la salida de la CNN (luego de reordenar):', tuple(z_a.shape))
print('Dimensiones de la salida del BiLSTM:', tuple(h_a.shape))
print('Diferencia de la CNN en t=2:', torch.max(torch.abs(z_a[0, 1] - z_b[0, 1])).item())
print('Diferencia hacia adelante en t=2:', dif_adelante)
print('Diferencia hacia atrás en t=2:', dif_atras)

In [ ]:
resumen_a = ### <-- COMPLETAR
resumen_b = ### <-- COMPLETAR
dif_resumen = torch.max(torch.abs(resumen_a - resumen_b)).item()
print('Diferencia máxima entre resúmenes de las primeras dos posiciones:', dif_resumen)

diferencias = torch.abs(h_a[0] - h_b[0])
frames = np.arange(1, 5)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(frames, diferencias[:, :32].max(dim=1).values.numpy(), marker='o', label='Dirección hacia adelante')
ax.plot(frames, diferencias[:, 32:].max(dim=1).values.numpy(), marker='s', label='Dirección hacia atrás')
ax.set_xticks(frames)
ax.set_xlabel('Frame $t$')
ax.set_ylabel('Diferencia máxima (abs)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()